In [3]:
import os
from pathlib import Path
import json
import pandas as pd
from mri_data import utils
import nibabel as nib
from statistics import mean
import pyperclip
import sys

from core.experiment import Experiment
from core.dataset import Dataset
sys.path.append("/home/srs-9/Projects/prl_project")
from helpers.shell_interface import command, run_if_missing

from helpers.paths import load_config

In [2]:
dataroot = Path("/media/smbshare/srs-9/prl_project/data-roi_train1")
inf_root = Path("/media/smbshare/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output")
subject_sessions = pd.read_csv("/home/srs-9/Projects/prl_project/resources/subject-sessions.csv", index_col="sub")

In [10]:
run_dir = Path("/media/smbshare/srs-9/prl_project/training/roi_train1_segresnet")
label_config = load_config(run_dir / "label_config.json")
ds = Dataset(label_config["dataset_name"])
exp = Experiment.from_run_dir(run_dir, ds)

In [ ]:
path1 = Path("/media/smbshare/srs-9/prl_project/data-roi_train1")
path2 = Path("/media/smbshare/srs-9/prl_project/data")

path1.home(

PosixPath('/home/srs-9')

In [3]:
with open("datalist.json", 'r') as f:
    datalist = json.load(f)
    
test_data = datalist["testing"]
prl_data = []
lesion_data = []
for scan in test_data:
    scan = {k: Path(v) for k,v in scan.items()}
    if  "prl" in scan['label'].name:
        scan['inference'] = inf_root / scan['label'].relative_to(dataroot).with_name("flair.phase_ensemble.nii.gz")
        prl_data.append(scan)
    else:
        scan['inference'] = inf_root / scan['label'].relative_to(dataroot).with_name("flair.phase_ensemble.nii.gz")
        lesion_data.append(scan)


In [6]:
scan

{'image': PosixPath('/media/smbshare/srs-9/prl_project/data-roi_train1/sub1215-20180429/3/flair.phase.nii.gz'),
 'label': PosixPath('/media/smbshare/srs-9/prl_project/data-roi_train1/sub1215-20180429/3/prl_label_final.nii.gz'),
 'inference': PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1215-20180429/3/flair.phase_ensemble.nii.gz')}

In [7]:
prl_dice_scores = []
for scan in prl_data:
    lab_data = nib.load(scan['label']).get_fdata()
    inf_data = nib.load(scan['inference']).get_fdata()
    prl_dice_scores.append([scan['image'].parent.name,
                            scan['image'].parent.parent.name,
                            utils.dice_score(lab_data, inf_data, seg1_val=2, seg2_val=2)])

In [8]:
prl_dice_scores

[['1', 'sub1131-20161209', np.float64(0.7368421052631579)],
 ['5', 'sub2041-20170515', np.float64(0.16414686825053995)],
 ['6', 'sub1348-20181009', np.float64(0.06451612903225806)],
 ['1', 'sub2131-20190117', np.float64(0.7192851824274014)],
 ['3', 'sub1215-20180429', np.float64(0.13574660633484162)]]

In [16]:
import statistics

statistics.mean(prl_dice_scores)

0.36410737826163975

In [20]:
# view_root = Path("Z:/")
view_root = Path("/media/smbshare")
for scan in prl_data:
    images = [scan['image'].with_name(im) for im in ["flair.nii.gz", "phase.nii.gz"]]
    images = [view_root/(im.relative_to("/media/smbshare")) for im in images]
    labels = [scan['label'], scan['inference']]
    labels = [view_root/(im.relative_to("/media/smbshare")) for im in labels]
    cmd = utils.open_itksnap_workspace_cmd(images, labels=labels)
    print(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub1131-20161209/1/flair.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub1131-20161209/1/phase.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub1131-20161209/1/prl_label_final.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1131-20161209/1/flair.phase_ensemble.nii.gz
itksnap -g /media/smbshare/srs-9/prl_project/data/sub2041-20170515/5/flair.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2041-20170515/5/phase.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2041-20170515/5/prl_label_final.nii.gz /media/smbshare/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub2041-20170515/5/flair.phase_ensemble.nii.gz
itksnap -g /media/smbshare/srs-9/prl_project/data/sub1348-20181009/6/flair.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub1348-20181009/6/phase.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub1348-20181009/6/prl_label_final.nii.gz /media/smbshare/srs-9/prl_

In [13]:
prl_dice_scores

[0.7368421052631579,
 0.16414686825053995,
 0.06451612903225806,
 0.7192851824274014,
 0.13574660633484162]

In [14]:
lesion_dice_scores = []
for scan in lesion_data:
    lab_data = nib.load(scan['label']).get_fdata()
    inf_data = nib.load(scan['inference']).get_fdata()
    lesion_dice_scores.append(utils.dice_score(lab_data, inf_data, seg1_val=2, seg2_val=2))

In [15]:
check_lesions = []
for i, (scan, dice) in enumerate(zip(lesion_data, lesion_dice_scores)):
    if dice is not None:
        check_lesions.append(scan)

In [16]:
view_root = Path("Z:/")
for scan in check_lesions:
    images = [scan['image'].with_name(im) for im in ["flair.nii.gz", "phase.nii.gz"]]
    images = [view_root/(im.relative_to("/media/smbshare")) for im in images]
    labels = [scan['label'], scan['inference']]
    labels = [view_root/(im.relative_to("/media/smbshare")) for im in labels]
    cmd = utils.open_itksnap_workspace_cmd(images, labels=labels)
    print(cmd)

itksnap -g Z:/srs-9/prl_project/data/sub1011-20180911/3/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1011-20180911/3/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1011-20180911/3/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1011-20180911/3/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1038-20161031/2/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1038-20161031/2/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1038-20161031/2/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1038-20161031/2/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1348-20181009/2/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1348-20181009/2/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1348-20181009/2/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1348-20181009/2/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub2131-20190117/2/flair.nii.gz -o Z:/srs-9/p

In [16]:
subid = 2026
sesid = subject_sessions.loc[subid, "ses"]
subject_root = dataroot / f"sub{subid}-{sesid}"
l = 2
images = [subject_root/f"{l}/phase.nii.gz", subject_root/f"{l}/flair.nii.gz"]
segs = [subject_root/f"{l}/prl_label_final.nii.gz"]
cmd = utils.open_itksnap_workspace_cmd(images, labels=segs)
print(cmd)
pyperclip.copy(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/phase.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/flair.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/prl_label_final.nii.gz


In [19]:
segmentations = [
    "prl_mask_def_prob_CH.nii.gz",
    "prl_mask_def_prob_SRS.nii.gz",
    "prl_mask_def_prob_LR.nii.gz",
]
subid = 2026
sesid = subject_sessions.loc[subid, "ses"]
subject_root = dataroot / f"sub{subid}-{sesid}"

for seg in segmentations:
    if (subject_root / seg).exists():
        break
images = [subject_root/"phase.nii.gz", subject_root/"flair.nii.gz"]
segs = [subject_root/seg, subject_root/"lstai_lesion_index.nii.gz"]
cmd = utils.open_itksnap_workspace_cmd(images, labels=segs)
print(cmd)
pyperclip.copy(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub2026-20160917/phase.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2026-20160917/flair.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2026-20160917/prl_mask_def_prob_SRS.nii.gz /media/smbshare/srs-9/prl_project/data/sub2026-20160917/lstai_lesion_index.nii.gz
